In [206]:
import pandas as pd
import numpy as np
import warnings

In [207]:
warnings.filterwarnings('ignore')

In [208]:
df = pd.read_csv('../data/raw/visualisation_df.csv').drop(columns=['Unnamed: 0', 'X', 'probability_of_failure'])
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment,one_year_prob
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN,0.014321
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0,0.008131
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN,0.015885
3,SOAD00862,Navaldia,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Rated,295.1,25.3,NaN,NaN,0,NaN,NaN,NaN,0.016265
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0,0.011027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20801,SOAD16536,Lyndrassia,No,Flood Risk Reduction,Rockfill,35.872,1.667,7411000.0,480.62681,84247.85257,...,Not Available,820.9,510.6,NaN,47.0,0,47.0,2.0,47.0,0.009078
20802,SOAD13145,Lyndrassia,No,Hydroelectric,Gravity,141.296,0.963,375000.0,405.40236,62491.43656,...,Not Available,850.2,176.6,62.9,52.0,0,52.0,5.0,52.0,0.008579
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0,0.010708
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0,0.007160


In [209]:
df = df.loc[df['primary_type'] == 'Earth']

In [210]:
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment,one_year_prob
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN,0.014321
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0,0.008131
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN,0.015885
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0,0.011027
5,SOAD02227,Lyndrassia,Yes,Recreation,Earth,14.332,0.127,NaN,0.06107,NaN,...,Unsatisfactory,11.5,733.7,8.2,27.0,0,27.0,27.0,27.0,0.014654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20798,SOAD11040,Navaldia,No,Hydroelectric,Earth,3.956,0.105,NaN,1499.77873,50184.37347,...,Satisfactory,850.1,163.8,83.8,59.0,0,59.0,3.0,2.0,0.010512
20799,SOAD12695,Navaldia,No,Hydroelectric,Earth,3.444,0.230,NaN,1539.68108,49808.84958,...,Satisfactory,770.7,331.4,87.1,59.0,0,59.0,3.0,2.0,0.008308
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0,0.010708
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0,0.007160


In [211]:
df['damage_loss'] = df['damage_loss'].fillna(0)
df['dam_repair_loss'] = df['dam_repair_loss'].fillna(0)
df['business_interruption_loss'] = df['business_interruption_loss'].fillna(0)

In [212]:
df['business_interruption_loss'].isna().sum()

0

In [213]:
df['total_loss'] = df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss']
df['expected_loss'] = df['one_year_prob'] * (df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss'])

### hazard

In [214]:
def calculate_hazard_rating_factor(df, hazard):
    curr_hazard_mean = (df[df['hazard'] == hazard].describe()['total_loss'].loc['50%'] + df[df['hazard'] == hazard].describe()['total_loss'].loc['50%']) / 2
    low_hazard_mean = (df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%'] + df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%']) / 2
    hazard_rating_factor = curr_hazard_mean/low_hazard_mean
    return hazard_rating_factor

In [215]:
hazard_low_rf = calculate_hazard_rating_factor(df, 'Low')
hazard_high_rf = calculate_hazard_rating_factor(df, 'High')
hazard_significant_rf = calculate_hazard_rating_factor(df, 'Significant')
hazard_undetermined_rf = calculate_hazard_rating_factor(df, 'Undetermined')

In [216]:
# Define mapping dictionary
hazard_mapping = {
    'Low': hazard_low_rf,
    'High': hazard_high_rf,
    'Significant': hazard_significant_rf,
    'Undetermined': hazard_undetermined_rf
}

# Create a new column using map()
df['hazard_rating_factor'] = df['hazard'].map(hazard_mapping)


In [217]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### Regulation

In [218]:
df['no_BI_loss'] = df['dam_repair_loss'] + df['damage_loss']

In [219]:
df['w'] = df['no_BI_loss'] / df['total_loss']
df['w'] = df['w'].fillna(1)
df['failure_rate'] = df['one_year_prob'] * df['w']
def calculate_regulation_rating_factor(df, region):
    regulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'Yes')]['failure_rate'])
    unregulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'No')]['failure_rate'])
    regulated_rating_factor = unregulated_region_failure_rate / regulated_region_failure_rate 
    return regulated_rating_factor

In [220]:
# Define mapping dictionary
regulated_mapping = {
    'Navaldia': calculate_regulation_rating_factor(df, 'Navaldia'),
    'Lyndrassia': calculate_regulation_rating_factor(df, 'Lyndrassia'),
    'Flumevale': calculate_regulation_rating_factor(df, 'Flumevale')
}

# Create a new column using map()
df['regulated_rating_factor'] = df['region'].map(regulated_mapping)


In [221]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### GDP

In [222]:
gdp_df = pd.read_excel('../data/raw/soaGDP.xlsx', sheet_name='2025 Nominal GDP')
pop_df = pd.read_csv('../data/raw/population10yrs.csv')

In [223]:
pop_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,45363514,7067855,39808697,92240066
1,2020,45502051,7097789,40175188,92775028
2,2021,45651175,7131024,40565887,93348086
3,2022,45599000,7157446,40953108,93709554
4,2023,45311937,7239138,42148205,94699280
5,2024,45161092,7267582,42526941,95150152
6,2025,45145328,7324559,43313896,95925972
7,2026,45259214,7353108,43693348,96378123
8,2027,45404777,7409970,44479504,97152512
9,2028,45473424,7438623,44859669,97605929


In [224]:
gdp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,4.671259e+06,534401.257379,3.779710e+06,8.985371e+06


In [225]:
for feature in gdp_df.columns:
    gdp_df[feature] = gdp_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)
    pop_df[feature] = pop_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)

In [226]:
gdp_pp_df = pd.DataFrame({
    'Year': gdp_df['Year']
})
# change here for gdp in different year
for feature in gdp_df.drop(columns=['Year']).columns:
    gdp_pp_df[feature] = gdp_df[feature].item()/pop_df.iloc[6][feature]

In [227]:
gdp_pp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,0.103472,0.07296,0.087263,0.09367


In [228]:
for feature in gdp_pp_df.drop(columns=['Year', 'Tarrodan']):
    gdp_pp_df[feature] = gdp_pp_df['Tarrodan']/gdp_pp_df[feature]

In [229]:
regulated_mapping = {
    'Navaldia': gdp_pp_df['Navaldia'].item(),
    'Lyndrassia': gdp_pp_df['Lyndrassia'].item(),
    'Flumevale': gdp_pp_df['Flumevale'].item()
}

# Create a new column using map()
df['gdp_rating_factor'] = df['region'].map(regulated_mapping)


### summary

In [230]:
df['total_rating_factor'] = df['hazard_rating_factor'] * df['regulated_rating_factor'] * df['gdp_rating_factor']

In [231]:
df[['hazard_rating_factor', 'regulated_rating_factor', 'gdp_rating_factor', 'total_rating_factor']].describe()

,hazard_rating_factor,regulated_rating_factor,gdp_rating_factor,total_rating_factor
count,19368.000000,19368.000000,19368.000000,19368.000000
mean,2.791123,0.859706,1.132780,2.601651
std,2.677251,0.533684,0.138102,3.531527
min,0.095631,0.074239,0.905271,0.058796
25%,1.000000,0.572772,1.073417,0.614823
50%,1.000000,0.572772,1.073417,1.884630
75%,3.970874,1.467953,1.283849,1.884630
max,7.224272,1.467953,1.283849,13.615079


### premium

In [232]:
'''Flu_BI_df = df.loc[(df['region'] == 'Flumevale') & (df['business_interruption_loss'] != 0)]
Lyn_BI_df = df.loc[(df['region'] == 'Lyndrassia') & (df['business_interruption_loss'] != 0)]
Nav_BI_df = df.loc[(df['region'] == 'Navaldia') & (df['business_interruption_loss'] != 0)]
Flu_no_BI_df = df.loc[(df['region'] == 'Flumevale') & (df['business_interruption_loss'] == 0)]
Lyn_no_BI_df = df.loc[(df['region'] == 'Lyndrassia') & (df['business_interruption_loss'] == 0)]
Nav_no_BI_df = df.loc[(df['region'] == 'Navaldia') & (df['business_interruption_loss'] == 0)]'''


"Flu_BI_df = df.loc[(df['region'] == 'Flumevale') & (df['business_interruption_loss'] != 0)]\nLyn_BI_df = df.loc[(df['region'] == 'Lyndrassia') & (df['business_interruption_loss'] != 0)]\nNav_BI_df = df.loc[(df['region'] == 'Navaldia') & (df['business_interruption_loss'] != 0)]\nFlu_no_BI_df = df.loc[(df['region'] == 'Flumevale') & (df['business_interruption_loss'] == 0)]\nLyn_no_BI_df = df.loc[(df['region'] == 'Lyndrassia') & (df['business_interruption_loss'] == 0)]\nNav_no_BI_df = df.loc[(df['region'] == 'Navaldia') & (df['business_interruption_loss'] == 0)]"

In [233]:
'''for sub_df in [Flu_BI_df, Lyn_BI_df, Nav_BI_df, Flu_no_BI_df, Lyn_no_BI_df, Nav_no_BI_df]:
    print(sub_df.shape)'''

'for sub_df in [Flu_BI_df, Lyn_BI_df, Nav_BI_df, Flu_no_BI_df, Lyn_no_BI_df, Nav_no_BI_df]:\n    print(sub_df.shape)'

In [234]:
'''def remove_extreme_value(df, lower_percentile, upper_percentile, feature):
    loc_df = df.loc[(df[feature] > df[feature].quantile(lower_percentile)) & (df[feature] < df[feature].quantile(upper_percentile))]
    return loc_df
'''

'def remove_extreme_value(df, lower_percentile, upper_percentile, feature):\n    loc_df = df.loc[(df[feature] > df[feature].quantile(lower_percentile)) & (df[feature] < df[feature].quantile(upper_percentile))]\n    return loc_df\n'

In [235]:
'''combined_df = pd.DataFrame(columns=df.columns)
for sub_df in [Flu_BI_df, Lyn_BI_df, Nav_BI_df, Flu_no_BI_df, Lyn_no_BI_df, Nav_no_BI_df]:
    curr_df = remove_extreme_value(df=sub_df, lower_percentile=0.1, upper_percentile=0.8, feature='total_loss')
    combined_df = pd.concat([combined_df, curr_df], ignore_index=True)'''

"combined_df = pd.DataFrame(columns=df.columns)\nfor sub_df in [Flu_BI_df, Lyn_BI_df, Nav_BI_df, Flu_no_BI_df, Lyn_no_BI_df, Nav_no_BI_df]:\n    curr_df = remove_extreme_value(df=sub_df, lower_percentile=0.1, upper_percentile=0.8, feature='total_loss')\n    combined_df = pd.concat([combined_df, curr_df], ignore_index=True)"

### dam repair loss

In [236]:
df['dam_repair_loss_re'] = df['dam_repair_loss'] * 0.4
df['dam_repair_loss_owner'] = df['dam_repair_loss'] * 0.6 * 0.8
df['dam_repair_loss_business'] = df['dam_repair_loss'] * 0.6 * 0.1
df['dam_repair_loss_people'] = df['dam_repair_loss'] * 0.6 * 0.1

In [237]:
df['TP_loss_re'] = df['damage_loss'] * 0.4
df['TP_loss_owner'] = df['damage_loss'] * 0.6 * 0.6
df['TP_loss_business'] = df['damage_loss'] * 0.6 * 0.2
df['TP_loss_people'] = df['damage_loss'] * 0.6 * 0.2

In [238]:
df['BI_loss_re'] = df['business_interruption_loss'] * 0.4
df['BI_loss_owner'] = df['business_interruption_loss'] * 0.6 * 0.6
df['BI_loss_business'] = df['business_interruption_loss'] * 0.6 * 0.2

In [239]:
df['loss_re'] = df['dam_repair_loss_re'] + df['TP_loss_re'] + df['BI_loss_re']
df['loss_owner'] = df['dam_repair_loss_owner'] + df['TP_loss_owner'] + df['BI_loss_owner']
df['loss_business'] = df['dam_repair_loss_business'] + df['TP_loss_business'] + df['BI_loss_business']
df['loss_people'] = df['dam_repair_loss_people'] + df['TP_loss_people']

In [ ]:
for part in ['re', 'owner', 'business', 'people']:
    df[f'premium_{part}'] = df[f'loss_{part}'] * df['one_year_prob']

In [241]:
df[['premium_re', 'premium_owner', 'premium_business', 'premium_people']].describe()

,premium_re,premium_owner,premium_business,premium_people
count,19368.000000,19368.000000,19368.000000,19368.000000
mean,2.657479,2.635691,0.675264,0.655189
std,2.874850,2.932330,0.717255,0.710673
min,0.000000,0.000000,0.000000,0.000000
25%,0.399207,0.425520,0.085120,0.068219
50%,1.597969,1.507329,0.425595,0.406946
75%,4.049287,3.906344,1.040058,1.019183
max,21.455588,22.393721,4.894831,4.815006


In [242]:
for part in ['re', 'owner', 'business', 'people']:
    df[f'adjusted_premium_{part}'] = df[f'premium_{part}'] * df['total_rating_factor']

In [243]:
df[['adjusted_premium_re', 'adjusted_premium_owner', 'adjusted_premium_business', 'adjusted_premium_people']].describe()

,adjusted_premium_re,adjusted_premium_owner,adjusted_premium_business,adjusted_premium_people
count,19368.000000,19368.000000,19368.000000,19368.000000
mean,10.028227,9.829481,2.606430,2.560946
std,20.323602,20.268493,5.232405,5.170779
min,0.000000,0.000000,0.000000,0.000000
25%,0.266285,0.282486,0.058471,0.046163
50%,1.762852,1.718976,0.433873,0.417565
75%,8.057676,7.524240,2.236255,2.206639
max,249.597726,262.993885,55.701351,54.826931


In [244]:
for region in df['region'].value_counts().keys():
    print(df[(df['region'] == region) & (df['business_interruption_loss'] == 0)]['premium_owner'].sum()/np.square(((df['region'] == region) & (df['business_interruption_loss'] == 0)).sum()))
    

0.0005582321554773181
0.0003294384182032021
0.004061325130985245


In [245]:
for region in df['region'].value_counts().keys():
    print(df[(df['region'] == region) & (df['business_interruption_loss'] != 0)]['premium_owner'].sum()/np.square(((df['region'] == region) & (df['business_interruption_loss'] != 0)).sum()))

0.0006764357586060651
0.0010320128678863606
0.0018438544189826613


In [246]:
df['total_loss'].sum() * (df['one_year_prob'].mean()) / np.square(df['id'].count())

0.00020002442926294874

In [247]:
for region in df['region'].value_counts().keys():
    print(df[df['region'] == region]['one_year_prob'].sum())

88.51798299084143
83.65163226770474
29.50482861449117
